# Transfer Learning for EEG Depression Detection
## EEGNet pre-trained → fine-tuned on MODMA 128-channel data

**Strategy:**
1. Use EEGNet architecture (designed specifically for EEG classification)
2. Pre-train on a larger public EEG dataset (BCI Competition IV Dataset 2a — motor imagery, 9 subjects × large recordings)
3. Fine-tune the feature extractor layers on MODMA depression data (53 subjects)
4. Subject-level LOO CV with majority vote across windows

**Why EEGNet works for transfer learning:**
- Designed for EEG — uses depthwise convolutions that respect channel structure
- Learns spatial filters (which channels matter) + temporal filters (which frequencies matter)
- Pre-training on any EEG task teaches the model general EEG signal structure
- Fine-tuning on depression adapts those features to the target task



In [ ]:
# Step 1 — Install dependencies
!pip install mne torch torchvision torchaudio scipy numpy pandas scikit-learn -q
print("Done")


In [ ]:
# Step 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_SAVE = Path("/content/drive/MyDrive/EDAIC")
EEG_DIR = Path("/content/drive/MyDrive/EEG_128channels_resting_lanzhou_2015")

import os
print(f"BASE_SAVE exists: {BASE_SAVE.exists()}")
print(f"EEG_DIR exists  : {EEG_DIR.exists()}")
print(f"Files in EDAIC  : {list(BASE_SAVE.glob('*.csv')) if BASE_SAVE.exists() else 'not found'}")


In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import signal as sp_signal, io as sio
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
warnings.filterwarnings("ignore")

# Device — GPU on Colab
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1. EEGNet Architecture

In [ ]:
class EEGNet(nn.Module):

    def __init__(self, n_channels=128, n_timepoints=2500,
                 n_classes=2, F1=8, D=2, F2=16,
                 dropout=0.5, freeze_backbone=False):
        super().__init__()
        self.freeze_backbone = freeze_backbone

        # Block 1: Temporal convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )

        # Block 2: Depthwise spatial convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1*D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 3: Separable convolution
        self.block3 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 16), padding=(0, 8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        # Compute output size
        dummy  = torch.zeros(1, 1, n_channels, n_timepoints)
        dummy  = self.block1(dummy)
        dummy  = self.block2(dummy)
        dummy  = self.block3(dummy)
        self.feat_dim = dummy.view(1, -1).shape[1]

        # Classifier head — replaced during fine-tuning
        self.classifier = nn.Linear(self.feat_dim, n_classes)

    def forward(self, x):
        # x: (B, 1, C, T)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def get_features(self, x):

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.view(x.size(0), -1)

    def freeze_backbone_layers(self):

        for param in self.block1.parameters(): param.requires_grad = False
        for param in self.block2.parameters(): param.requires_grad = False
        for param in self.block3.parameters(): param.requires_grad = False

    def unfreeze_all(self):

        for param in self.parameters(): param.requires_grad = True


model_test = EEGNet(n_channels=128, n_timepoints=2500)
dummy = torch.zeros(2, 1, 128, 2500)
out   = model_test(dummy)
print(f"EEGNet output shape: {out.shape}")
print(f"Feature dimension  : {model_test.feat_dim}")
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Parameters         : {n_params:,}")


## 2. Data Loading & Preprocessing

In [ ]:
FS          = 250
WINDOW_SAMP = 2500
STEP_SAMP   = 1250
N_CHANNELS  = 128

def load_mat(fpath: Path) -> np.ndarray | None:
    try:
        mat = sio.loadmat(str(fpath))
        for k, v in mat.items():
            if k.startswith('_') or k in ('samplingRate','Impedances_0'):
                continue
            if hasattr(v,'shape') and len(v.shape)==2:
                if (128 in v.shape or 129 in v.shape) and max(v.shape) > 2000:
                    arr = v.astype(np.float32)
                    if arr.shape[0] in (128,129) and arr.shape[1] > arr.shape[0]:
                        return arr[:128, :]
                    elif arr.shape[1] in (128,129) and arr.shape[0] > arr.shape[1]:
                        return arr.T[:128, :]
        return None
    except Exception as e:
        print(f"  [ERROR] {fpath.name}: {e}")
        return None


def preprocess(data: np.ndarray) -> np.ndarray:
    data = data.copy().astype(np.float64)
    data -= data.mean(axis=1, keepdims=True)
    b, a = sp_signal.butter(4, [1.0, 40.0], btype='bandpass', fs=FS)
    for ch in range(data.shape[0]):
        data[ch] = sp_signal.filtfilt(b, a, data[ch])
    std = data.std(axis=1, keepdims=True) + 1e-8
    return (data / std).astype(np.float32)


def make_windows(data: np.ndarray):
    wins = []
    s = 0
    while s + WINDOW_SAMP <= data.shape[1]:
        wins.append(data[:, s:s+WINDOW_SAMP])
        s += STEP_SAMP
    return np.stack(wins).astype(np.float32) if wins else None


def get_label(fname: str):
    stem   = Path(fname).stem
    digits = ''.join(filter(str.isdigit, stem[:8]))
    try:
        sid = int(digits)
        if str(sid).startswith('201'): return sid, 1
        if str(sid).startswith('203') or str(sid).startswith('202'): return sid, 0
    except: pass
    return None, None


class EEGDataset(Dataset):
    def __init__(self, windows, label):

        self.X = torch.from_numpy(windows).unsqueeze(1)
        self.y = label
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y


print("Loading MODMA 128-channel subjects...")
mat_files = sorted(EEG_DIR.glob("*.mat")) if EEG_DIR.exists() else []

if not mat_files:
    print(f"[ERROR] No .mat files found in {EEG_DIR}")
    print("Please upload the EEG_128channels_resting_lanzhou_2015 folder to Google Drive")
else:
    subjects = []
    for fpath in mat_files:
        sid, label = get_label(fpath.name)
        if sid is None: continue
        raw = load_mat(fpath)
        if raw is None: continue
        data = preprocess(raw)
        wins = make_windows(data)
        if wins is None or len(wins) == 0: continue
        subjects.append((sid, label, wins))

    n_mdd = sum(1 for _,l,_ in subjects if l==1)
    n_hc  = sum(1 for _,l,_ in subjects if l==0)
    print(f"Loaded: {len(subjects)} subjects ({n_mdd} MDD, {n_hc} HC)")
    print(f"Avg windows/subject: {np.mean([len(w) for _,_,w in subjects]):.1f}")


In [ ]:
def augment_window(window: torch.Tensor) -> torch.Tensor:
    """
    EEG augmentation for pre-training:
    - Gaussian noise
    - Channel dropout
    - Time shifting
    """
    w = window.clone()
    # Gaussian noise
    w += torch.randn_like(w) * 0.05
    # Random channel dropout (10% of channels)
    drop_mask = torch.rand(w.shape[0]) > 0.9
    w[drop_mask] = 0
    # Time shift (up to 125 samples = 0.5s)
    shift = np.random.randint(-125, 125)
    w = torch.roll(w, shift, dims=-1)
    return w


class AugmentedEEGDataset(Dataset):

    def __init__(self, windows_list, augment=True):
        # windows_list: list of (windows_array, label)
        self.windows = []
        self.labels  = []
        for wins, label in windows_list:
            for w in wins:
                self.windows.append(w)
                self.labels.append(label)
        self.augment = augment

    def __len__(self): return len(self.windows)

    def __getitem__(self, i):
        w = torch.from_numpy(self.windows[i]).unsqueeze(0)  # (1, 128, 2500)
        if self.augment:
            w = augment_window(w)
        return w, self.labels[i]


def pretrain(model, train_subjects, epochs=30, device=DEVICE):
    """Pre-train EEGNet on training subjects with augmentation."""
    train_data = [(w, l) for _, l, w in train_subjects]
    dataset    = AugmentedEEGDataset(train_data, augment=True)
    loader     = DataLoader(dataset, batch_size=32, shuffle=True,
                            num_workers=2, pin_memory=True)

    n_mdd = sum(1 for _, l in train_data if l==1)
    n_hc  = sum(1 for _, l in train_data if l==0)
    n_tot = n_mdd + n_hc
    weights   = torch.tensor([n_tot/(2*n_hc), n_tot/(2*n_mdd)],
                              dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = torch.tensor(yb, dtype=torch.long).to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch+1) % 10 == 0:
            print(f"    Pre-train epoch {epoch+1}/{epochs} loss={total_loss/len(loader):.4f}")
    return model


def finetune(model, train_subjects, epochs=20, lr=1e-4, device=DEVICE):
    """
    Fine-tune EEGNet on depression task.
    Strategy: freeze backbone first, train classifier, then unfreeze all.
    """
    train_data = [(w, l) for _, l, w in train_subjects]
    dataset    = AugmentedEEGDataset(train_data, augment=False)
    loader     = DataLoader(dataset, batch_size=32, shuffle=True,
                            num_workers=2, pin_memory=True)

    n_mdd = sum(1 for _, l in train_data if l==1)
    n_hc  = sum(1 for _, l in train_data if l==0)
    n_tot = n_mdd + n_hc
    weights   = torch.tensor([n_tot/(2*n_hc), n_tot/(2*n_mdd)*1.5],
                              dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # Phase 1: freeze backbone, train only classifier (10 epochs)
    model.freeze_backbone_layers()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr*10, weight_decay=1e-4)

    model.train()
    for epoch in range(min(10, epochs)):
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = torch.tensor(yb, dtype=torch.long).to(device)
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()

    # Phase 2: unfreeze all, fine-tune with low LR
    model.unfreeze_all()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                       T_max=epochs)

    best_loss  = float('inf')
    best_state = None
    patience   = 5; pat_cnt = 0

    for epoch in range(epochs):
        total_loss = 0
        model.train()
        for Xb, yb in loader:
            Xb = Xb.to(device)
            yb = torch.tensor(yb, dtype=torch.long).to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(loader)
        if avg_loss < best_loss:
            best_loss  = avg_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_cnt    = 0
        else:
            pat_cnt += 1
            if pat_cnt >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return model


def predict_subject(model, windows, device=DEVICE):
    """Majority vote across windows for subject-level prediction."""
    X = torch.from_numpy(windows).unsqueeze(1).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(dim=1).cpu().numpy()
    return int(np.bincount(preds, minlength=2).argmax())


print("Training functions defined.")


## 4. LOO CV — Pre-train → Fine-tune → Predict

In [ ]:
import copy

def compute_metrics(y_true, y_pred):
    return {
        "accuracy"   : float(accuracy_score(y_true, y_pred)),
        "UAR"        : float(recall_score(y_true, y_pred, average="macro")),
        "F1"         : float(f1_score(y_true, y_pred, average="weighted")),
        "sensitivity": float(recall_score(y_true, y_pred, pos_label=1)),
        "specificity": float(recall_score(y_true, y_pred, pos_label=0)),
    }

PRETRAIN_EPOCHS  = 30
FINETUNE_EPOCHS  = 25
RANDOM_STATE     = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Shuffle for balanced running metrics
rng = np.random.default_rng(RANDOM_STATE)
subjects_shuffled = list(subjects)
rng.shuffle(subjects_shuffled)

print(f"LOO CV — {len(subjects_shuffled)} folds")
print(f"Pre-train: {PRETRAIN_EPOCHS} epochs | Fine-tune: {FINETUNE_EPOCHS} epochs")
print()

all_preds  = []
all_labels = []
fold_rows  = []

for i, (val_sid, val_label, val_windows) in enumerate(subjects_shuffled):
    train_subs = [(s,l,w) for s,l,w in subjects_shuffled if s != val_sid]

    # Fresh model each fold
    model = EEGNet(n_channels=N_CHANNELS, n_timepoints=WINDOW_SAMP,
                   n_classes=2, dropout=0.5).to(DEVICE)

    # Pre-train on training subjects
    model = pretrain(model, train_subs, epochs=PRETRAIN_EPOCHS, device=DEVICE)

    # Fine-tune on depression task
    model = finetune(model, train_subs, epochs=FINETUNE_EPOCHS, device=DEVICE)

    # Predict held-out subject
    pred = predict_subject(model, val_windows, device=DEVICE)

    all_preds.append(pred)
    all_labels.append(val_label)

    if len(all_preds) >= 5:
        run_uar = recall_score(all_labels, all_preds, average="macro")
        run_acc = accuracy_score(all_labels, all_preds)
    else:
        run_uar = run_acc = float('nan')

    fold_rows.append({"fold":i+1, "subject":val_sid,
                      "true":val_label, "pred":pred})
    print(f"  Fold {i+1:2d}/{len(subjects_shuffled)}: "
          f"sid={val_sid} true={val_label} pred={pred} | "
          f"UAR={run_uar:.3f} Acc={run_acc:.3f}")

    # Free GPU memory
    del model
    torch.cuda.empty_cache()


## 5. Final Results

In [ ]:
m = compute_metrics(all_labels, all_preds)

print("=" * 60)
print("EEGNET TRANSFER LEARNING — MODMA 128-Channel EEG")
print("=" * 60)
print(f"  UAR         : {m['UAR']:.4f} ({m['UAR']*100:.2f}%)")
print(f"  Accuracy    : {m['accuracy']:.4f} ({m['accuracy']*100:.2f}%)")
print(f"  Sensitivity : {m['sensitivity']:.4f}")
print(f"  Specificity : {m['specificity']:.4f}")
print(f"  F1          : {m['F1']:.4f}")
print()
print(f"  Published baseline (Shi et al. 2020): Acc=72.25%")
print(f"  QSVC best result                    : UAR=0.7019")

if m['UAR'] > 0.70:
    print(f"\n  ✓ Matches or beats QSVC result!")
if m['accuracy'] > 0.7225:
    print(f"  ✓ Beats published baseline!")

# Save
import json
pd.DataFrame(fold_rows).to_csv(BASE_SAVE / "modma_eegnet_loo_results.csv", index=False)
with open(BASE_SAVE / "modma_eegnet_summary.json", "w") as f:
    json.dump({"model": "EEGNet Transfer Learning", **m,
               "pretrain_epochs": PRETRAIN_EPOCHS,
               "finetune_epochs": FINETUNE_EPOCHS}, f, indent=2)

print("\nSaved to Google Drive.")
